Loading beam patterns into healpy
---------------------------------

A common use case for diffuse maps is to simulate an antenna's response to the sky, by computing the antenna temperature (i.e. the beam-weighted average of the sky temperature).

Here are some code snippets that may be helpful.

### Converting a beam pattern in (theta, phi) to healpix

This uses [pyshtools](https://shtools.github.io/SHTOOLS/) to evaluate a beam pattern on the healpix grid.

```python
# Load data
beam_data = load_data(filename)

# Load data into pysh
# You may need to pad your data if it only covers one hemisphere
beam_data = np.pad(beam_data, pad_width=((0, 0), (0, 180)), mode='edge')
B = pysh.SHGrid.from_array(beam_data)

# Generate a healpix grid
nside = 128
pix_idx = np.arange(hp.nside2npix(nside))
lon, lat = hp.pix2ang(nside, pix_idx, lonlat=True)

# Convert pysh grid to spherical harmonics
# The definition is different to that used in healpy
E_clm = E.expand(lmax_calc=99, normalization='ortho', csphase=-1)

# Evaluate the pysh spherical harmonics at healpix grid locations
hpx_map = pysh.expand.MakeGridPoint(E_clm.coeffs, lat=lat, lon=lon, norm=4)  # norm=4 == ortho
```


### Rotating the healpix beam pattern toward observing direction

This rotates maps in spherical harmonic space, using healpy rotators.

```python
# Set the astropy sky coordinate for the desired beam centre (ra/dec)
sky_coord = SkyCoord(...) 
ll = sky_coord.galactic.l.to('deg').value
bb = sky_coord.galactic.b.to('deg').value
ra = sky_coord.icrs.ra.to('deg').value
dec = sky_coord.icrs.dec.to('deg').value

# If the map needs to be rotated away from alignment with N-E, apply first rotation.
# Healpy uses right-hand rule rotations. A positive rotation about the X-axis 
# will tilt the north pole toward -y.
if rot_angle > 0:
    r0r = hp.rotator.Rotator(rot=[rot_angle + 90, 0, 0], coord=None, eulertype='X')
    hpx_map_rotated = r0r.rotate_map_alms(hpx_map)

# Next rotation: from (theta,phi) to (alt, az)
# To convert (theta, phi) antenna coords to (alt, az) astro coords:
#     az = 90 - phi
# To convert to Galactic coords (l, b) also requires 90 degree shift:
#     b = 90 - phi
r0 = hp.rotator.Rotator(rot=[0, 90, 0], coord=None, eulertype='Y')
hpx_map_rotated_astro = r0.rotate_map_alms(hpx_map_rotated_astro)

# Third rotation - to sky coordinate (RA / DEC)
r1 = hp.Rotator(rot=[ra, dec], coord=['G', 'C'], inv=True)
hpx_map_rotated_skycoord = r1.rotate_map_alms(hpx_map_rotated_astro)

# Create a masked array so we can mask below horizon
mask = np.ones(shape=hpx.shape[0], dtype='bool')

# Now create below horizon mask
# ang2vec takes longitude and latitude in degrees when lonlat=True
vec = hp.ang2vec(ll, bb, lonlat=True)
ipix = hp.query_disc(nside=nside, vec=vec, radius=0.99 * np.pi / 2)  # Mask very edge pixels
mask[ipix] = 0

hpx_map_rotated_masked = np.ma.array(hpx_map_rotated_astro)
hpx.mask = mask
hpx.fill_value = 0   # Can also use np.nan or 1e20
```

### Dealing with polarized beam patterns

```python

# First, load data for Etheta and Ephi

# Jones matrix consists of Etheta, Ephi for X and Y pol
J = np.array(((Etheta_x, Ephi_x), (Etheta_y, Ephi_y)))

# Convert to a linear polarization coherency matrix by taking the outer product
C_mat = np.matmul(J, np.conjugate(J))

# Extract real-valued arrays 
C = {
    'xx':   np.real(C_mat[0, 0]),
    're_xy': np.real(C_mat[0, 1]),
    'im_xy': np.imag(C_mat[0, 1]),
    'yy':    np.real(C_mat[1, 1]),
}

# Convert linear polarization to Stokes parameters
S = {
    'I': C['xx'] + C['yy'],
    'Q': C['xx'] - C['yy'],
    'U': 2 * C['re_xy'],
    'V': 2 * C['im_xy']
}
```